In [2]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score

In [151]:
df=pd.read_csv("HousePricePrediction.csv")

### Features Enginnering 

- dealing with Null values  .isnull.sum()  and .dropna()
- columns selection for X and Y .drop()
- getdummies for catogories

In [152]:
df2=df.dropna()

In [153]:
# making our target values slightly linear-friendly  with log function log1p (1+x) so that value don't crash when log(0) 

In [154]:
X1=df2.drop(columns=["Id","BsmtFinSF2","SalePrice"])
y1=np.log1p(df2["SalePrice"])

In [155]:
X1.head(2)

,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,TotalBsmtSF
0,60,RL,8450,Inside,1Fam,5,2003,2003,VinylSd,856.0
1,20,RL,9600,FR2,1Fam,8,1976,1976,MetalSd,1262.0


In [156]:
# getdummies for cateogrial data 
X2=pd.get_dummies(X1,columns=["MSZoning","LotConfig","BldgType","Exterior1st"],dtype=int,drop_first=True)

In [157]:
# train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(
    X2,y1,test_size=0.2,random_state=42
)

In [158]:
# columns are high now so it can cause problems of Underfitting as there are Different Scaling Data -- PolynomialFeatures
poly = PolynomialFeatures(degree=2,include_bias=False)
X_poly_train =poly.fit_transform(X_train)
X_poly_test = poly.transform(X_test)

In [159]:
# scaling the data through StandardScaler
scaler=StandardScaler()
X_train_scaled= scaler.fit_transform(X_poly_train)
X_test_scaled= scaler.transform(X_poly_test)

In [160]:
# model form
from sklearn.linear_model import LassoCV
a=[0.001,0.01,0.1,1,10,20,50,100]
model=LassoCV(
    alphas=a,
    cv=5,
    max_iter=1000,
    tol=0.1
)
model.fit(X_train_scaled,Y_train)
Y_pred=model.predict(X_test_scaled)
Y_pred_train = model.predict(X_train_scaled)

In [161]:
# Y(output) to log1p log(1+x) --> to reverse  expo1m  exp(e^x-1) undo of log
true_ytrain=np.expm1(Y_train)
train_pred=np.expm1(Y_pred_train)
true_pred=np.expm1(Y_pred)
true_y= np.expm1(Y_test)
print(f'accuracy : {r2_score(true_y,true_pred)*100}')
print(f'accuracy training: {r2_score(true_ytrain,train_pred)*100}')
print("best alpha :", model.alpha_)

accuracy : 65.10409953402593
accuracy training: 75.93915526745528
best alpha : 0.001
